In [1]:
"""
Week 2: Exploratory Data Analysis & Statistical Modeling Pipeline
Construction Project Performance Analytics
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 150

# 1. Load Cleaned Dataset (from Week 1)
try:
    df = pd.read_csv("construction_projects_cleaned.csv")
except FileNotFoundError:
    # Fallback synthetic generation for standalone execution
    from construction_pipeline import (
        clean_and_prepare_pipeline,
        generate_construction_dataset,
    )

    df = clean_and_prepare_pipeline(generate_construction_dataset(500))

print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")

# =========================================================================
# 2. EXPLORATORY VISUALIZATIONS (4 Key Figures)
# =========================================================================

# --- FIGURE 1: Correlation Heatmap ---
plt.figure(figsize=(10, 8))
numeric_cols = [
    "Planned_Duration_Days",
    "Actual_Duration_Days",
    "Delay_Days",
    "Planned_Budget_USD",
    "Actual_Cost_USD",
    "Cost_Overrun_Pct",
    "Weather_Delay_Days",
    "Safety_Incidents_Count",
    "Subcontractor_Score",
    "Cost_Performance_Index_CPI",
    "Schedule_Performance_Index_SPI",
]
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Figure 1: Construction Metric Correlation Heatmap", fontsize=14, weight="bold", pad=12)
plt.tight_layout()
plt.savefig("fig1_correlation_heatmap.png")
plt.close()

# --- FIGURE 2: Delay vs Subcontractor Reliability ---
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="Subcontractor_Score",
    y="Delay_Days",
    hue="Project_Type",
    style="Project_Type",
    s=70,
    alpha=0.8,
)
sns.regplot(
    data=df,
    x="Subcontractor_Score",
    y="Delay_Days",
    scatter=False,
    color="black",
    line_kws={"linestyle": "--", "linewidth": 2},
)
plt.title("Figure 2: Schedule Delay vs Subcontractor Reliability", fontsize=13, weight="bold")
plt.xlabel("Subcontractor Reliability Score (0.0 to 1.0)")
plt.ylabel("Realized Delay (Days)")
plt.tight_layout()
plt.savefig("fig2_delay_vs_reliability.png")
plt.close()

# --- FIGURE 3: Cost Overrun Distribution by Location Zone ---
plt.figure(figsize=(9, 5))
sns.violinplot(
    data=df,
    x="Location_Zone",
    y="Cost_Overrun_Pct",
    palette="Set2",
    inner="quartile",
    cut=0,
)
plt.title("Figure 3: Cost Overrun % Distribution by Location Zone", fontsize=13, weight="bold")
plt.xlabel("Location Zone")
plt.ylabel("Cost Overrun Percentage (%)")
plt.tight_layout()
plt.savefig("fig3_cost_overrun_violin.png")
plt.close()

# --- FIGURE 4: Compounding Delays: Weather & Safety ---
plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    df["Weather_Delay_Days"],
    df["Delay_Days"],
    c=df["Safety_Incidents_Count"],
    cmap="viridis",
    s=df["Cost_Overrun_Pct"] * 3,
    alpha=0.7,
    edgecolors="w",
)
cbar = plt.colorbar(scatter)
cbar.set_label("Safety Incidents Count")
plt.title(
    "Figure 4: Schedule Delay vs Weather Impact (Sized by Cost Overrun %)",
    fontsize=13,
    weight="bold",
)
plt.xlabel("Weather Delay Days")
plt.ylabel("Total Project Delay (Days)")
plt.tight_layout()
plt.savefig("fig4_hazard_interaction.png")
plt.close()

print("[SUCCESS] All 4 visual artifacts generated and saved as PNG.")

# =========================================================================
# 3. STATISTICAL MODELING PIPELINE
# =========================================================================

# Define features and target
feature_cols = [
    "Planned_Duration_Days",
    "Planned_Budget_USD",
    "Material_Cost_Ratio",
    "Labor_Hours_Logged",
    "Weather_Delay_Days",
    "Safety_Incidents_Count",
    "Subcontractor_Score",
]
X = pd.get_dummies(
    df[feature_cols + ["Project_Type", "Location_Zone"]],
    columns=["Project_Type", "Location_Zone"],
    drop_first=True,
)
y = df["Delay_Days"]

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# --- Model 1: Ordinary Least Squares (OLS) ---
ols = LinearRegression()
ols.fit(X_train, y_train)
y_pred_ols = ols.predict(X_test)

# --- Model 2: Ridge Regularized Regression ---
ridge = RidgeCV(alphas=np.logspace(-2, 3, 50))
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

# --- Model 3: Gradient Boosting Baseline ---
gbm = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)
y_pred_gbm = gbm.predict(X_test)


# Evaluation Function
def evaluate_model(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return {"Model": name, "R2": round(r2, 3), "RMSE": round(rmse, 2), "MAE": round(mae, 2)}


benchmark_df = pd.DataFrame(
    [
        evaluate_model("OLS Linear Regression", y_test, y_pred_ols),
        evaluate_model("Ridge Regularized Regression", y_test, y_pred_ridge),
        evaluate_model("Gradient Boosting (GBM)", y_test, y_pred_gbm),
    ]
)

print("\n--- Model Benchmark Performance ---")
print(benchmark_df.to_string(index=False))

Dataset Loaded Successfully: 500 rows, 22 columns


C:\Users\adity\AppData\Local\Temp\ipykernel_14316\944948815.py:88: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(
C:\Users\adity\anaconda3\Lib\site-packages\matplotlib\collections.py:1115: RuntimeWarning: invalid value encountered in sqrt
  scale = np.sqrt(self._sizes) * dpi / 72.0 * self._factor


[SUCCESS] All 4 visual artifacts generated and saved as PNG.

--- Model Benchmark Performance ---
                       Model    R2  RMSE   MAE
       OLS Linear Regression 0.068 24.24 19.71
Ridge Regularized Regression 0.649 14.86 11.22
     Gradient Boosting (GBM) 0.584 16.19 12.25
